# Subclustering epithelials cells

In [ ]:
library(Signac)
library(Seurat)
library(rtracklayer)
library(dplyr)
library(Matrix)
library(ggplot2)
library(cowplot)
require(biomaRt)
library(DoubletFinder)
library(RColorBrewer)
library(stringr)
set.seed(1234)

In [ ]:
my_c <- c('#fe864a', '#c9d179', '#95d0fc', '#fbdd7e', '#388004', '#c760ff', '#76cd26', '#ffbacd', '#d6b4fc',
        '#cfaf7b','#8b88f8', '#acc2d9', '#894585', '#82cbb2', '#d99b82', '#fdee73', '#5170d7', '#ffd8b1', 
        '#b7e1a1', '#82a67d', '#b7fffa', '#b790d4', '#f5674a', '#C79FEF', '#DAA520', '#A0522D')

# Data loading

 This genes are the ones used to perform the aligment on CellRanger using the costumized genome reference */projects/F202408336CPCAA0/Analysis/Mouse/multiome/multiome_data/10X_ref_genome_plus_eGFP_CreERT*. It contains teh CreERT2 gene and e GFP gene.

In [ ]:
annotation <- import('genes2.gtf')

In [ ]:
annotation$gene_biotype[1:1780455] = annotation$gene_type[1:1780455]
seqlevelsStyle(annotation) <- "UCSC" #Support for other than Ensembl seqlevel style

In [ ]:
annotation$tx_id = annotation$transcript_id

In [ ]:
#load fragments data
P11_sample1_frags <- 'P11_sample1_atac_fragments.tsv.gz'

P11_sample2_frags <- 'P11_sample2_atac_fragments.tsv.gz'

P11_sample3_frags <- 'P11_sample3_atac_fragments.tsv.gz'

P60_sample1_frags <- 'P60_sample1_atac_fragments.tsv.gz'

P60_sample2_frags <- 'P60_sample2_atac_fragments.tsv.gz'

P60_TTR_sample3_frags <- 'P60_TTR_sample3_atac_fragments.tsv.gz'

In [ ]:
sample_names <- c('P11_sample1', 'P11_sample2', 'P11_sample3', 'P60_sample1', 'P60_sample2', 'P60_TTR_sample3')

In [ ]:
frags_path <- c(P11_sample1_frags, P11_sample2_frags, P11_sample3_frags, P60_sample1_frags, P60_sample2_frags, 
           P60_TTR_sample3_frags)

In [ ]:
raw <- readRDS('Merged_multiome.rds')

In [ ]:
filtered <- readRDS('Integrated_multiome_clusters_joint_linked_cell_type.rds')

In [ ]:
raw

In [ ]:
filtered

# Subset data into only Epithelial cells

In [ ]:
Epithelial_filt <- subset(filtered, subset = predicted.id == 'Epithelial cells')

In [ ]:
Epithelial_filt

In [ ]:
epithelial <- subset(raw, cells = rownames(Epithelial_filt@meta.data))

In [ ]:
epithelial

In [ ]:
frags <- Fragments(epithelial)

In [ ]:
Fragments(epithelial) <- NULL

In [ ]:
for (i in 1:length(frags)){
    frags[[i]] <- UpdatePath(frags[[i]], new.path = frags_path[[i]])
}

In [ ]:
Fragments(epithelial) <- frags

In [ ]:
saveRDS(epithelial, 'Epithelial_multiome.rds')

In [ ]:
epithelial <- readRDS('Epithelial_multiome.rds')

In [ ]:
epithelial@assays

In [ ]:
unique(epithelial$Sample)

In [ ]:
unique(epithelial$Age)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
epithelial$nucleosome_group <- ifelse(epithelial$nucleosome_signal > 2.0, 'NS > 2.0', 'NS < 2.0')

frag_hist <- FragmentHistogram(object = epithelial, group.by = 'nucleosome_group', region = 'chr1-1-10000000')

frag_hist

ggsave(file="Epithelial_frag_hist.svg", plot=frag_hist, width=8, height=4)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(epithelial, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="Epithelial_all_vln_prefilter.svg", plot=vln, width=15, height=8)


In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(epithelial, subset = Sample == sample)
    vln <- VlnPlot(object = s, features = c("nCount_RNA", "nFeature_RNA", "nCount_ATAC", "nFeature_ATAC", "TSS.enrichment", "nucleosome_signal", "pct_mito", "pct_hb"), 
                  ncol = 8, pt.size = 0)
    
    plot_name = paste("", sample, "_vln_prefilter.svg")

    print(vln)
    
    ggsave(file=plot_name, plot=vln, width=30, height=5)
    
    
}

# Filtering

In [ ]:
table(epithelial$orig.ident)

In [ ]:
DefaultAssay(epithelial) <- "RNA"

Since we already have the peaks assay we can delete the ATAC assay, which allows to save compuattional resources 

In [ ]:
samples.list <- SplitObject(epithelial, split.by = 'Sample')

In [ ]:
samples.list

In [ ]:
vars_to_regress <- c('Sample')

In [ ]:
epithelial <- epithelial %>% 
                            NormalizeData(normalization.method = "LogNormalize", scale.factor = 10000) %>%
                            FindVariableFeatures(selection.method = "vst", nfeatures=4000, ) %>%
                            ScaleData() %>%
                            RunPCA() 

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
pca <- ElbowPlot(epithelial, ndims = 50)

pca

ggsave(file="Epithelial_bff_pca.svg", plot=pca, width=6, height=4)

In [ ]:
epithelial <- RunUMAP(epithelial, dims = 1:13, reduction = 'pca', n.neighbors = 15)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dpl <- DimPlot(epithelial, reduction = 'umap', group.by = 'Sample', pt.size = 0.5, raster = TRUE, cols = my_c)

dpl

ggsave(file="Epithelial_bff_RNA_UMAP.svg", plot=dpl, width=8, height=4)

**NOTE**:The functions *get_density* and *DensityScatter* are from the packages Signac V 1.10, since we are using the version 1.4 we copied only the necessary function to plot the wanted plots.

In [ ]:
get_density <- function(x, y, ...) {
  if (!requireNamespace("MASS", quietly = TRUE)) {
    stop("Please install MASS: install.packages('MASS')")
  }
  if (!is.numeric(x = x) | !is.numeric(x = y)) {
      stop("Must supply numeric values")
  }
  dens <- MASS::kde2d(x = x, y = y, ...)
  ix <- findInterval(x = x, vec = dens$x)
  iy <- findInterval(x = y, vec = dens$y)
  ii <- cbind(ix, iy)
  return(dens$z[ii])
}

In [ ]:
DensityScatter <- function(
    object,
    x,
    y,
    log_x = FALSE,
    log_y = FALSE,
    quantiles = NULL
) {
    md <- object[[]]
    if (!(x %in% colnames(x = md))) {
        stop(x, " not found")
    }
    if (!(y %in% colnames(x = md))) {
        stop(y, " not found")
    }
    log10p <- function(x) {
        return(log10(x = x + 1))
    }
    null_fn <- function(x) {
        return(x)
    }
    logfnx <- ifelse(test = log_x, yes = log10p, no = null_fn)
    logfny <- ifelse(test = log_y, yes = log10p, no = null_fn)
    # logfny <- null_fn
    md$Density <- get_density(
        x = logfnx(md[[x]]),
        y = logfny(md[[y]]),
        n = 1000,
        h = c(1,1)
    )
    md <- md[order(md$Density), ]
    # quantiles
    use_quantile <- FALSE
    if (!is.null(x = quantiles)) {
        use_quantile <- TRUE
        # set default if TRUE passed
        if (is.logical(x = quantiles)) {
            if (quantiles) {
                quantiles <- c(5, 10, 90, 95)
            } else {
                use_quantile <- FALSE
            }
        }
        # make sure integers between 0 and 100
        if (!(all(quantiles >= 0) & all(quantiles <= 100))) {
            warning("Quantile values must be between 0 and 100",
                    immediate. = TRUE)
            use_quantile <- FALSE
        }
        if (any(sapply(X = quantiles, FUN = function(x) x %% 1 != 0))) {
            warning("Quantile values must be integers",
                    immediate. = TRUE)
            use_quantile <- FALSE
        }
    }
    if (use_quantile) {
        # convert to string
        quantiles <- paste0(quantiles, '%')

        # quantiles for x and y
        x_quant <- quantile(x = md[[x]], probs = seq(0, 1, 0.01))
        y_quant <- quantile(x = md[[y]], probs = seq(0, 1, 0.01))
        xlines <- x_quant[quantiles]
        ylines <- y_quant[quantiles]

        # round
        xlines <- round(x = xlines, digits = 2)
        ylines <- round(x = ylines, digits = 2)
    }        
    p <- ggplot(
           data = md,
           mapping = aes(x = .data[[x]], y = .data[[y]], color = .data[["Density"]])
         ) +
           geom_point(size = 1) +
           scale_color_viridis_c(option = "B") +
           theme_bw()
    if (log_x) {
        p <- p + scale_x_log10()
    }
    if (log_y) {
        p <- p + scale_y_log10()
    }
    if (use_quantile) {
        p <- p +
          geom_vline(xintercept = unname(obj = xlines), color = "red") +
          geom_hline(yintercept = unname(obj = ylines), color = "red") +
          labs(title = "Quantiles",
               subtitle = paste0(x, ": ", paste0(names(x = xlines), ":",
                                                 xlines, collapse = " "), "\n",
                                 y, ": ", paste0(names(x = ylines), ":",
                                                 ylines, collapse = " ")))
    }
    return(p)
}

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(epithelial, subset = Sample == sample)
    scatterplt <- DensityScatter(object = s, x = 'nCount_RNA', y = 'nFeature_RNA', log_x = TRUE, quantiles = TRUE)
    
    plot_name = paste("", sample, "Densenty_scatter_prefilter_nodbl_RNA.svg")

    options(repr.plot.width = 8, repr.plot.height = 6)
    print(scatterplt)
    
    ggsave(file=plot_name, plot=scatterplt, width=8, height=6)
    
    
}

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(epithelial, subset = Sample == sample)
    scatterplt <- DensityScatter(object = s, x = 'nCount_ATAC', y = 'TSS.enrichment', log_x = TRUE, quantiles = TRUE)
    
    plot_name = paste("", sample, "Densenty_scatter_prefilter_nodbl.svg")

    options(repr.plot.width = 8, repr.plot.height = 6)
    print(scatterplt)
    
    ggsave(file=plot_name, plot=scatterplt, width=8, height=6)
    
    
}

In [ ]:
nCount_RNA_max <- c(12500, 12500, 16000, 30000, 50000, 5000)

nCount_RNA_min <- c(250, 250, 250, 250, 250, 250)

nFeature_RNA_max <- c(6000, 6000, 6000, 6500, 9500, 2500)

nFeature_RNA_min <- c(250, 250, 250, 250, 250, 250)

nCount_ATAC_max <- c(16000, 17500, 20000, 25000, 60000, 9000)

nCount_ATAC_min <- c(400, 400, 400, 450, 600, 200)

nFeature_ATAC_max <- c(7500, 8000, 9000, 10000, 23000, 4000)

nFeature_ATAC_min <- c(250, 250, 250, 250, 250, 250)

nucleosome_signal_max <- c(2.0, 2.0, 2.0, 2.0, 2.0, 2.0)

TSS.enrichment_min <- c(2.0, 2.0, 2.0, 2.0, 2.0, 2.0)

pct_mito_max <- c(0.3, 0.3, 0.3, 1.3, 0.7, 2.0)

pct_hb_max <- c(0.075, 0.075, 0.06, 0.04, 0.04, 0.04)


In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
cells_to_keep <- c()

In [ ]:
for (i in 1:length(sample_names)){
    sample <- sample_names[i]
    print(sample)

    s <- subset(epithelial, subset = Sample == sample)
    #Idents(s) <- 'orig.ident'
    print(s)
    
    print(nCount_RNA_max[i])
    print(nCount_ATAC_max[i])
    print(nCount_RNA_min[i])
    print(nCount_ATAC_min[i])
    print(nFeature_RNA_max[i])
    print(nFeature_ATAC_max[i])
    print(nFeature_RNA_min[i])
    print(nFeature_ATAC_min[i])
    print(nucleosome_signal_max[i])
    print(TSS.enrichment_min[i])
    print(pct_mito_max[i])
    print(pct_hb_max[i])
      
      
    s <-  subset(s, subset=nCount_RNA < nCount_RNA_max[i] &
                      nCount_ATAC < as.numeric(nCount_ATAC_max[i]) &
                      nCount_RNA > as.numeric(nCount_RNA_min[i]) &
                      nCount_ATAC > as.numeric(nCount_ATAC_min[i]) &
                      nFeature_RNA < as.numeric(nFeature_RNA_max[i]) &
                      nFeature_ATAC < as.numeric(nFeature_ATAC_max[i]) &
                      nFeature_RNA > as.numeric(nFeature_RNA_min[i]) &
                      nFeature_ATAC > as.numeric(nFeature_ATAC_min[i]) &
                      nucleosome_signal < as.numeric(nucleosome_signal_max[i]) &
                      TSS.enrichment > as.numeric(TSS.enrichment_min[i]) &
                      pct_mito < as.numeric(pct_mito_max[i]) &
                      pct_hb < as.numeric(pct_hb_max[i]))
    
    cells_to_keep <- c(cells_to_keep, colnames(s))
    
    print(length(colnames(s)))
    print(length(cells_to_keep))
    
    vln <- VlnPlot(object = s, features = c("nCount_RNA", "nFeature_RNA", "nCount_ATAC", "nFeature_ATAC", "TSS.enrichment", "nucleosome_signal", "pct_mito", "pct_hb"), 
                  ncol = 8, pt.size = 0)
    
    print(vln)

    plot_name = paste("", sample, "_vln_afterfilter.svg")

    
    ggsave(file=plot_name, plot=vln, width=30, height=5)
    
}

In [ ]:
length(cells_to_keep)

In [ ]:
epithelial

In [ ]:
Epithelial_filt <- subset(epithelial, cells = cells_to_keep)

In [ ]:
length(colnames(Epithelial_filt))

In [ ]:
length(unique(colnames(Epithelial_filt)))

In [ ]:
table(Epithelial_filt$orig.ident)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(Epithelial_filt, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="Epithelial_all_vln_after_filter.svg", plot=vln, width=15, height=8)


In [ ]:
saveRDS(Epithelial_filt, 'Epithelial_filt_multiome.rds')

# Call peaks

In [ ]:
Epithelial_filt <- readRDS('Epithelial_filt_multiome.rds')

In [ ]:
Epithelial_filt

In [ ]:
DefaultAssay(Epithelial_filt) <- 'ATAC'

In [ ]:
# call peaks using MACS2
peaks <- CallPeaks(object=Epithelial_filt, 
                outdir='callpeaks', cleanup=FALSE)

In [ ]:
# remove peaks on nonstandard chromosomes and in genomic blacklist regions
peaks <- keepStandardChromosomes(peaks, pruning.mode = "coarse")
peaks <- subsetByOverlaps(peaks, ranges = blacklist_mm10, invert = TRUE)

In [ ]:
macs2_counts <- FeatureMatrix(fragments = Fragments(Epithelial_filt), features = peaks, cells = colnames(Epithelial_filt))

In [ ]:
Epithelial_filt[['peaks']] <- CreateChromatinAssay(counts = macs2_counts, sep = c('-', '-'), fragments = Fragments(Epithelial_filt), annotation = annotation)

In [ ]:
Epithelial_filt@assays

In [ ]:
DefaultAssay(Epithelial_filt) <- 'peaks'

In [ ]:
Epithelial_filt[['ATAC']] <- NULL

In [ ]:
saveRDS(Epithelial_filt, 'Epithelial_Epithelial_filt_multiome_peaks.rds')

# FRiP

In [ ]:
Epithelial_filt <- readRDS('Epithelial_Epithelial_filt_multiome_peaks.rds')

In [ ]:
Epithelial_filt <- FRiP(Epithelial_filt, assay = 'peaks', total.fragments = "nCount_ATAC", col.name = 'FRiP', verbose = TRUE)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 12, repr.plot.res = 300)

vln <- VlnPlot(Epithelial_filt, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_peaks', 'nFeature_peaks', 'nucleosome_signal', 'TSS.enrichment', 'FRiP', 'pct_mito', 'pct_hb'), 
        ncol=3, pt.size=0, col = my_c) 

vln

ggsave(file="Epithelial_Epithelial_filt_vln_pre_filter_frip.svg", plot=vln, width=15, height=12)


In [ ]:
saveRDS(Epithelial_filt, 'Epithelial_Epithelial_filt_multiome_peaks.rds')

# Downstream analysis

In [ ]:
Epithelial_filt <- readRDS('Epithelial_Epithelial_filt_multiome_peaks.rds')

In [ ]:
DefaultAssay(Epithelial_filt) <- "RNA"

## RNA

In [ ]:
Epithelial_filt@assays

In [ ]:
table(Epithelial_filt$orig.ident)

In [ ]:
vars_to_regress <- c('Sample')

In [ ]:
Epithelial_filt <- Epithelial_filt %>% 
                            NormalizeData(normalization.method = "LogNormalize", scale.factor = 10000) %>%
                            FindVariableFeatures(selection.method = "vst", nfeatures=4000, ) %>%
                            ScaleData(vars.to.regress = vars_to_regress) %>%
                            RunPCA() 

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
pca <- ElbowPlot(Epithelial_filt, ndims = 50)

pca

ggsave(file="Epithelial_aff_pca.svg", plot=pca, width=6, height=4)

In [ ]:
Epithelial_filt <- RunUMAP(Epithelial_filt, dims = 1:13, reduction = 'pca', n.neighbors = 15, reduction.name="umap_RNA")

In [ ]:
unique(Epithelial_filt$Age)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
before_umap_s <- DimPlot(Epithelial_filt, reduction = 'umap_RNA', group.by = 'Sample', pt.size = 0.5, raster = TRUE, cols = my_c)

before_umap_s

ggsave(file="Epithelial_aff_Sample_UMAP.svg", plot=before_umap_s, width=8, height=4)


options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
before_umap_a <- DimPlot(Epithelial_filt, reduction = 'umap_RNA', group.by = 'Age', pt.size = 0.5, raster = TRUE, cols = my_c[2:3])

before_umap_a

ggsave(file="Epithelial_aff_Age_UMAP.svg", plot=before_umap_a, width=6, height=4)

options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
before_umap_a <- DimPlot(Epithelial_filt, reduction = 'umap_RNA', group.by = 'Batch', pt.size = 0.5, raster = TRUE, cols = my_c[2:3])

before_umap_a

ggsave(file="Epithelial_aff_Batch_UMAP.svg", plot=before_umap_a, width=6, height=4)

### Integration with Harmony

In [ ]:
library(harmony)

In [ ]:
integrated <- RunHarmony(Epithelial_filt, group.by.vars = 'Sample', reduction.save = 'harmonyRNA')

In [ ]:
integrated

In [ ]:
integrated <- RunUMAP(integrated, reduction = 'harmonyRNA', dims = 1:13, n.neighbors = 15, reduction.name="harmony_RNA")

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
after_umap_s <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Sample', pt.size = 0.5, raster = TRUE, cols = my_c)

after_umap_s

ggsave(file="Harmony_RNA_sample_UMAP.svg", plot=after_umap_s, width=8, height=4)


options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
after_umap_a <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Age', pt.size = 0.5, raster = TRUE, cols = my_c[2:3])

after_umap_a

ggsave(file="Harmony_RNA_Age_UMAP.svg", plot=after_umap_a, width=6, height=4)

options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
after_umap_b <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Batch', pt.size = 0.5, raster = TRUE, cols = my_c[2:3])

after_umap_b

ggsave(file="Harmony_RNA_Batch_UMAP.svg", plot=after_umap_b, width=6, height=4)


In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="orig.ident", reduction="harmony_RNA", shuffle=TRUE, cols=my_c, ncol=3, pt.size=0.5, label=FALSE)

dpl

ggsave(file="Harmony_each_sample_UMAP.svg", plot=dpl, width=24, height=12)

### Clustering

In [ ]:
integrated

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
integrated <- integrated %>% FindNeighbors(reduction = 'harmonyRNA', dims = 1:13) %>%
                                FindClusters(resolution = c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 300)
grid <- plot_grid(DimPlot(integrated, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.1', cols = my_c, raster = TRUE), 
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.2', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.3', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.4', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.5', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.6', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.7', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.8', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.9', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.1', cols = my_c, raster = TRUE))
grid

ggsave(file="Epithelial_integrated_clusters_all.svg", plot=grid, width=15, height=30)

In [ ]:
saveRDS(integrated, 'Epithelial_integrated_multiome_peaks_clusters.rds')

# Peaks

In [ ]:
integrated <- readRDS('Epithelial_integrated_multiome_peaks_clusters.rds')

In [ ]:
DefaultAssay(integrated) <- "peaks"

In [ ]:
integrated <- FindTopFeatures(integrated, assay = "peaks", min.cutoff = 10)

In [ ]:
integrated <- RunTFIDF(integrated, assay="peaks")

In [ ]:
integrated <- RunSVD(integrated, assay="peaks")

In [ ]:
options(repr.plot.width=6, repr.plot.height=4, repr.plot.res = 200)
dpc <- DepthCor(integrated, n=50)

dpc

ggsave(file="Merged_depthCorr.svg", plot=dpc, width=6, height=4)

In [ ]:
options(repr.plot.width=6, repr.plot.height=4, repr.plot.res = 200)
ebl <- ElbowPlot(integrated, reduction = "lsi", n=50)

ebl

ggsave(file="Merged_depthCorr_ebl.svg", plot=ebl, width=6, height=4)

In [ ]:
integrated <- FindNeighbors(integrated, reduction = 'lsi', dims = 2:12, graph.name="snn_peaks")

In [ ]:
integrated <- RunUMAP(integrated, dims = 2:12, reduction = 'lsi', reduction.name="umap_peaks")

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by="Sample", reduction="umap_peaks", pt.size = 0.5, raster = TRUE, cols = my_c, label=FALSE)

dpl 

ggsave(file="Merged_peaks_Sample_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="orig.ident", reduction="umap_peaks", shuffle=TRUE, cols=my_c, ncol=3, pt.size=1.5, raster = TRUE, label=FALSE)

dpl

ggsave(file="Merged_peaks_each_sample_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=7, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by="Batch", reduction="umap_peaks", pt.size = 0.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Merged_peaks_Batch_UMAP.svg", plot=dpl, width=7, height=4)

In [ ]:
options(repr.plot.width=7, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by="Age", reduction="umap_peaks", pt.size = 0.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Merged_peaks_Age_UMAP.svg", plot=dpl, width=7, height=4)

### Integration with Harmony

In [ ]:
library(harmony)

In [ ]:
integrated <- RunHarmony(integrated, group.by.vars = 'Sample', reduction = 'lsi', assay.use = 'peaks', reduction.save = 'harmonypeaks', project.dim = FALSE)

In [ ]:
# re-compute the UMAP using corrected LSI embeddings
integrated <- RunUMAP(integrated, dims = 2:12, reduction = 'harmonypeaks', reduction.name = 'harmony_peaks')

In [ ]:
options(repr.plot.width=8, repr.plot.height=5)
dpl <- DimPlot(integrated, group.by = 'Sample', pt.size = 0.5, raster=TRUE, cols=my_c, reduction = 'harmony_peaks')

dpl

ggsave(file="Epithelial_integrated_peaks_Sample_UMAP.svg", plot=dpl, width=8, height=5)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="orig.ident", reduction="harmony_peaks", shuffle=TRUE, cols=my_c, ncol=3, pt.size=0.5, raster = TRUE, label=FALSE)

dpl

ggsave(file="Epithelial_integrated_peaks_each_Sample_UMAP.svg", plot=dpl, width=24, height=12)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)
dpl <- DimPlot(integrated, group.by="Batch", reduction="harmony_peaks", pt.size = 0.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Epithelial_integrated_peaks_Batch_UMAP.svg", plot=dpl, width=7, height=5)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)
dpl <- DimPlot(integrated, group.by="Age", reduction="harmony_peaks", pt.size = 0.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Epithelial_integrated_peaks_Age_UMAP.svg", plot=dpl, width=7, height=5)

### Clustering

In [ ]:
DefaultAssay(integrated) <- 'peaks'

In [ ]:
integrated

In [ ]:
integrated <- integrated %>% FindNeighbors(reduction = 'harmonypeaks', dims = 2:12) %>%
                                FindClusters(resolution = c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 300)
grid <- plot_grid(DimPlot(integrated, reduction = "harmony_peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.1', cols = my_c, raster = TRUE), 
                  DimPlot(integrated, reduction = "harmony_peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.2', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.3', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.4', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.5', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.6', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.7', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.8', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.9', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_peaks", pt.size=1.5, group.by = 'peaks_snn_res.1', cols = my_c, raster = TRUE))
grid

ggsave(file="Epithelial_integrated_peaks_clusters_all.svg", plot=grid, width=15, height=30)

In [ ]:
saveRDS(integrated, 'Epithelial_integrated_multiome_peaks_clusters.rds')

# Cell type annotation

In [ ]:
integrated <- readRDS('Epithelial_integrated_multiome_peaks_clusters.rds')

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
integrated <- SetIdent(integrated, value = 'RNA_snn_res.1')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(integrated, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_peaks', 'nFeature_peaks', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

In [ ]:
marker_genes <- c('Folr1', 'Clic6', 'Krt18', 'Ttr', 'Dcn', 'Itga8', 'Col3a1', 'Flt1', 'Plvap', 'Pecam1', 'Cd86', 'Tank', 
                  'Cd83', 'Dscam', 'Ptprz1', 'Gria2')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 14, repr.plot.res = 300)
vln <- VlnPlot(object = integrated, features = marker_genes, pt.size=0.5, cols = my_c, ncol = 3)

vln

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 30, repr.plot.res = 200)
ftp <- FeaturePlot(integrated, reduction = 'harmony_RNA', features = marker_genes, min.cutoff = 0, max.cutoff = 4, label.size = 2, ncol = 3, pt.size = 1.5, raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 

# Remove clusters of doublets

In [ ]:
integrated <- subset(integrated, , idents = c(14), invert = TRUE)

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "RNA_snn_res.1", pt.size=1.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

ggsave(file="Epithelial_integrated_dbl_Removed_clusters_RNA.svg", plot=dpt, width=7, height=5)

In [ ]:
integrated

In [ ]:
Epithelial_noDBL <- subset(epithelial, cells=rownames(integrated@meta.data))

In [ ]:
Epithelial_noDBL

In [ ]:
saveRDS(Epithelial_noDBL, 'Epithelial_noDBL_multiome.rds')

# Redo the analysis without doublets manually removed

In [ ]:
Epithelial_noDBL <- readRDS( 'Epithelial_noDBL_multiome.rds')

In [ ]:
Epithelial_noDBL

In [ ]:
unique(Epithelial_noDBL$Sample)

In [ ]:
unique(Epithelial_noDBL$Age)

In [ ]:
DefaultAssay(Epithelial_noDBL) <- 'ATAC'

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
Epithelial_noDBL$nucleosome_group <- ifelse(Epithelial_noDBL$nucleosome_signal > 2.0, 'NS > 2.0', 'NS < 2.0')

frag_hist <- FragmentHistogram(object = Epithelial_noDBL, group.by = 'nucleosome_group', region = 'chr1-1-10000000')

frag_hist

ggsave(file="Epithelial_noDBL_frag_hist.svg", plot=frag_hist, width=8, height=4)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(Epithelial_noDBL, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="Epithelial_noDBL_all_vln_prefilter.svg", plot=vln, width=15, height=8)


In [ ]:
DefaultAssay(Epithelial_noDBL) <- 'RNA'

In [ ]:
DefaultAssay(Epithelial_noDBL) <- 'RNA'

options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(Epithelial_noDBL, subset = Sample == sample)
    vln <- VlnPlot(object = s, features = c("nCount_RNA", "nFeature_RNA", "nCount_ATAC", "nFeature_ATAC", "TSS.enrichment", "nucleosome_signal", "pct_mito", "pct_hb"), 
                  ncol = 8, pt.size = 0)
    
    plot_name = paste("", sample, "_noDBL_vln_prefilter.svg")

    print(vln)
    
    ggsave(file=plot_name, plot=vln, width=30, height=5)
    
    
}

# Filtering

In [ ]:
table(Epithelial_noDBL$orig.ident)

In [ ]:
DefaultAssay(Epithelial_noDBL) <- "RNA"

In [ ]:
samples.list <- SplitObject(Epithelial_noDBL, split.by = 'Sample')

In [ ]:
samples.list

In [ ]:
vars_to_regress <- c('Sample')

In [ ]:
Epithelial_noDBL <- Epithelial_noDBL %>% 
                            NormalizeData(normalization.method = "LogNormalize", scale.factor = 10000) %>%
                            FindVariableFeatures(selection.method = "vst", nfeatures=4000, ) %>%
                            ScaleData() %>%
                            RunPCA() 

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
pca <- ElbowPlot(Epithelial_noDBL, ndims = 50)

pca

ggsave(file="Epithelial_noDBL_pca.svg", plot=pca, width=6, height=4)

In [ ]:
Epithelial_noDBL <- RunUMAP(Epithelial_noDBL, dims = 1:15, reduction = 'pca', n.neighbors = 15)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dpl <- DimPlot(Epithelial_noDBL, reduction = 'umap', group.by = 'Sample', pt.size = 1.5, raster = TRUE, cols = my_c)

dpl

ggsave(file="Epithelial_noDBL_RNA_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
get_density <- function(x, y, ...) {
  if (!requireNamespace("MASS", quietly = TRUE)) {
    stop("Please install MASS: install.packages('MASS')")
  }
  if (!is.numeric(x = x) | !is.numeric(x = y)) {
      stop("Must supply numeric values")
  }
  dens <- MASS::kde2d(x = x, y = y, ...)
  ix <- findInterval(x = x, vec = dens$x)
  iy <- findInterval(x = y, vec = dens$y)
  ii <- cbind(ix, iy)
  return(dens$z[ii])
}

In [ ]:
DensityScatter <- function(
    object,
    x,
    y,
    log_x = FALSE,
    log_y = FALSE,
    quantiles = NULL
) {
    md <- object[[]]
    if (!(x %in% colnames(x = md))) {
        stop(x, " not found")
    }
    if (!(y %in% colnames(x = md))) {
        stop(y, " not found")
    }
    log10p <- function(x) {
        return(log10(x = x + 1))
    }
    null_fn <- function(x) {
        return(x)
    }
    logfnx <- ifelse(test = log_x, yes = log10p, no = null_fn)
    logfny <- ifelse(test = log_y, yes = log10p, no = null_fn)
    # logfny <- null_fn
    md$Density <- get_density(
        x = logfnx(md[[x]]),
        y = logfny(md[[y]]),
        n = 1000,
        h = c(1,1)
    )
    md <- md[order(md$Density), ]
    # quantiles
    use_quantile <- FALSE
    if (!is.null(x = quantiles)) {
        use_quantile <- TRUE
        # set default if TRUE passed
        if (is.logical(x = quantiles)) {
            if (quantiles) {
                quantiles <- c(5, 10, 90, 95)
            } else {
                use_quantile <- FALSE
            }
        }
        # make sure integers between 0 and 100
        if (!(all(quantiles >= 0) & all(quantiles <= 100))) {
            warning("Quantile values must be between 0 and 100",
                    immediate. = TRUE)
            use_quantile <- FALSE
        }
        if (any(sapply(X = quantiles, FUN = function(x) x %% 1 != 0))) {
            warning("Quantile values must be integers",
                    immediate. = TRUE)
            use_quantile <- FALSE
        }
    }
    if (use_quantile) {
        # convert to string
        quantiles <- paste0(quantiles, '%')

        # quantiles for x and y
        x_quant <- quantile(x = md[[x]], probs = seq(0, 1, 0.01))
        y_quant <- quantile(x = md[[y]], probs = seq(0, 1, 0.01))
        xlines <- x_quant[quantiles]
        ylines <- y_quant[quantiles]

        # round
        xlines <- round(x = xlines, digits = 2)
        ylines <- round(x = ylines, digits = 2)
    }        
    p <- ggplot(
           data = md,
           mapping = aes(x = .data[[x]], y = .data[[y]], color = .data[["Density"]])
         ) +
           geom_point(size = 1) +
           scale_color_viridis_c(option = "B") +
           theme_bw()
    if (log_x) {
        p <- p + scale_x_log10()
    }
    if (log_y) {
        p <- p + scale_y_log10()
    }
    if (use_quantile) {
        p <- p +
          geom_vline(xintercept = unname(obj = xlines), color = "red") +
          geom_hline(yintercept = unname(obj = ylines), color = "red") +
          labs(title = "Quantiles",
               subtitle = paste0(x, ": ", paste0(names(x = xlines), ":",
                                                 xlines, collapse = " "), "\n",
                                 y, ": ", paste0(names(x = ylines), ":",
                                                 ylines, collapse = " ")))
    }
    return(p)
}

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(Epithelial_noDBL, subset = Sample == sample)
    scatterplt <- DensityScatter(object = s, x = 'nCount_RNA', y = 'nFeature_RNA', log_x = TRUE, quantiles = TRUE)
    
    plot_name = paste("", sample, "Densenty_scatter_prefilter_nodbl_RNA.svg")
    #vlns_pre <- c(vlns_pre, vln)
    options(repr.plot.width = 8, repr.plot.height = 6)
    print(scatterplt)
    
    ggsave(file=plot_name, plot=scatterplt, width=8, height=6)
    
    
}

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(Epithelial_noDBL, subset = Sample == sample)
    scatterplt <- DensityScatter(object = s, x = 'nCount_ATAC', y = 'TSS.enrichment', log_x = TRUE, quantiles = TRUE)
    
    plot_name = paste("", sample, "Densenty_scatter_prefilter_nodbl.svg")

    options(repr.plot.width = 8, repr.plot.height = 6)
    print(scatterplt)
    
    ggsave(file=plot_name, plot=scatterplt, width=8, height=6)
    
    
}

In [ ]:
nCount_RNA_max <- c(12500, 12500, 16000, 30000, 45000, 6000)

nCount_RNA_min <- c(250, 250, 250, 250, 250, 250)

nFeature_RNA_max <- c(5000, 5000, 6000, 6500, 8000, 3000)

nFeature_RNA_min <- c(250, 250, 250, 250, 250, 250)

nCount_ATAC_max <- c(15000, 19000, 20000, 25000, 60000, 8000)

nCount_ATAC_min <- c(400, 400, 400, 450, 600, 200)

nFeature_ATAC_max <- c(8000, 7900, 9000, 10000, 20000, 3500)

nFeature_ATAC_min <- c(250, 250, 250, 250, 250, 250)

nucleosome_signal_max <- c(2.0, 2.0, 2.0, 2.0, 2.0, 2.0)

TSS.enrichment_min <- c(2.0, 2.0, 2.0, 2.0, 2.0, 2.0)

pct_mito_max <- c(0.3, 0.3, 0.3, 1.3, 0.7, 1.2)

pct_hb_max <- c(0.075, 0.075, 0.06, 0.05, 0.05, 0.05)

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
cells_to_keep <- c()

In [ ]:
for (i in 1:length(sample_names)){
    sample <- sample_names[i]
    print(sample)

    s <- subset(Epithelial_noDBL, subset = Sample == sample)

    print(s)
    
    print(nCount_RNA_max[i])
    print(nCount_ATAC_max[i])
    print(nCount_RNA_min[i])
    print(nCount_ATAC_min[i])
    print(nFeature_RNA_max[i])
    print(nFeature_ATAC_max[i])
    print(nFeature_RNA_min[i])
    print(nFeature_ATAC_min[i])
    print(nucleosome_signal_max[i])
    print(TSS.enrichment_min[i])
    print(pct_mito_max[i])
    print(pct_hb_max[i])
      
      
    s <-  subset(s, subset=nCount_RNA < nCount_RNA_max[i] &
                      nCount_ATAC < as.numeric(nCount_ATAC_max[i]) &
                      nCount_RNA > as.numeric(nCount_RNA_min[i]) &
                      nCount_ATAC > as.numeric(nCount_ATAC_min[i]) &
                      nFeature_RNA < as.numeric(nFeature_RNA_max[i]) &
                      nFeature_ATAC < as.numeric(nFeature_ATAC_max[i]) &
                      nFeature_RNA > as.numeric(nFeature_RNA_min[i]) &
                      nFeature_ATAC > as.numeric(nFeature_ATAC_min[i]) &
                      nucleosome_signal < as.numeric(nucleosome_signal_max[i]) &
                      TSS.enrichment > as.numeric(TSS.enrichment_min[i]) &
                      pct_mito < as.numeric(pct_mito_max[i]) &
                      pct_hb < as.numeric(pct_hb_max[i]))
    
    cells_to_keep <- c(cells_to_keep, colnames(s))
    
    print(length(colnames(s)))
    print(length(cells_to_keep))
    
    vln <- VlnPlot(object = s, features = c("nCount_RNA", "nFeature_RNA", "nCount_ATAC", "nFeature_ATAC", "TSS.enrichment", "nucleosome_signal", "pct_mito", "pct_hb"), 
                  ncol = 8, pt.size = 0)
    
    print(vln)

    plot_name = paste("", sample, "_noDBL_vln_afterfilter.svg")

    
    ggsave(file=plot_name, plot=vln, width=30, height=5)
    
}

In [ ]:
Epithelial_noDBL_filt <- subset(Epithelial_noDBL, cells = cells_to_keep)

In [ ]:
table(Epithelial_noDBL_filt$orig.ident)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(Epithelial_noDBL_filt, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="Epithelial_noDBL_all_vln_after_filter.svg", plot=vln, width=15, height=8)

In [ ]:
saveRDS(Epithelial_noDBL_filt, 'Epithelial_noDBL_filt_multiome.rds')

# Call peaks

In [ ]:
Epithelial_noDBL_filt <- readRDS('Epithelial_noDBL_filt_multiome.rds')

In [ ]:
Epithelial_noDBL_filt

In [ ]:
DefaultAssay(Epithelial_noDBL_filt) <- 'ATAC'

In [ ]:
# call peaks using MACS2
peaks <- CallPeaks(object=Epithelial_noDBL_filt, 
                outdir='callpeaks', cleanup=FALSE)

In [ ]:
# remove peaks on nonstandard chromosomes and in genomic blacklist regions
peaks <- keepStandardChromosomes(peaks, pruning.mode = "coarse")
peaks <- subsetByOverlaps(peaks, ranges = blacklist_mm10, invert = TRUE)

In [ ]:
macs2_counts <- FeatureMatrix(fragments = Fragments(Epithelial_noDBL_filt), features = peaks, cells = colnames(Epithelial_noDBL_filt))

In [ ]:
Epithelial_noDBL_filt[['peaks']] <- CreateChromatinAssay(counts = macs2_counts, sep = c('-', '-'), fragments = Fragments(Epithelial_noDBL_filt), 
                                             annotation = annotation)

In [ ]:
DefaultAssay(Epithelial_noDBL_filt) <- 'peaks'

In [ ]:
Epithelial_noDBL_filt[['ATAC']] <- NULL

# FRiP

In [ ]:
Epithelial_noDBL_filt <- FRiP(Epithelial_noDBL_filt, assay = 'peaks', total.fragments = "nCount_ATAC", col.name = 'FRiP', verbose = TRUE)

In [ ]:


options(repr.plot.width = 15, repr.plot.height = 12, repr.plot.res = 300)

vln <- VlnPlot(Epithelial_noDBL_filt, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_peaks', 'nFeature_peaks', 'nucleosome_signal', 'TSS.enrichment', 'FRiP', 'pct_mito', 'pct_hb'), 
        ncol=3, pt.size=0, col = my_c) 

vln

ggsave(file="Epithelial_Epithelial_noDBL_filt_vln_pre_filter_frip.svg", plot=vln, width=15, height=12)


In [ ]:
saveRDS(Epithelial_noDBL_filt, 'Epithelial_noDBL_Epithelial_filt_multiome_peaks.rds')

# Downstream analysis

In [ ]:
Epithelial_noDBL_filt <- readRDS('Epithelial_noDBL_Epithelial_filt_multiome_peaks.rds')

In [ ]:
DefaultAssay(Epithelial_noDBL_filt) <- "RNA"

## RNA

In [ ]:
Epithelial_noDBL_filt@assays

In [ ]:
table(Epithelial_noDBL_filt$orig.ident)

In [ ]:
vars_to_regress <- c('Sample')

In [ ]:
Epithelial_noDBL_filt <- Epithelial_noDBL_filt %>% 
                            NormalizeData(normalization.method = "LogNormalize", scale.factor = 10000) %>%
                            FindVariableFeatures(selection.method = "vst", nfeatures=4000, ) %>%
                            ScaleData(vars.to.regress = vars_to_regress) %>%
                            RunPCA() 

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
pca <- ElbowPlot(Epithelial_noDBL_filt, ndims = 50)

pca

ggsave(file="Epithelial_noDBL_aff_pca.svg", plot=pca, width=6, height=4)

In [ ]:
Epithelial_noDBL_filt <- RunUMAP(Epithelial_noDBL_filt, dims = 1:13, reduction = 'pca', n.neighbors = 15, reduction.name="umap_RNA")

In [ ]:
unique(Epithelial_noDBL_filt$Age)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
before_umap_s <- DimPlot(Epithelial_noDBL_filt, reduction = 'umap_RNA', group.by = 'Sample', pt.size = 1.5, raster = TRUE, cols = my_c)

before_umap_s

ggsave(file="Epithelial_noDBL_aff_Sample_UMAP.svg", plot=before_umap_s, width=8, height=4)


options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
before_umap_a <- DimPlot(Epithelial_noDBL_filt, reduction = 'umap_RNA', group.by = 'Age', pt.size = 1.5, raster = TRUE, cols = my_c[2:3])

before_umap_a

ggsave(file="Epithelial_noDBL_aff_Age_UMAP.svg", plot=before_umap_a, width=6, height=4)

options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
before_umap_a <- DimPlot(Epithelial_noDBL_filt, reduction = 'umap_RNA', group.by = 'Batch', pt.size = 1.5, raster = TRUE, cols = my_c[2:3])

before_umap_a

ggsave(file="Epithelial_noDBL_aff_Batch_UMAP.svg", plot=before_umap_a, width=6, height=4)

### Integration with Harmony

In [ ]:
library(harmony)

In [ ]:
integrated_noDBL <- RunHarmony(Epithelial_noDBL_filt, group.by.vars = 'Sample', reduction.save = 'harmonyRNA')

In [ ]:
integrated_noDBL

In [ ]:
integrated_noDBL <- RunUMAP(integrated_noDBL, reduction = 'harmonyRNA', dims = 1:13, n.neighbors = 15, reduction.name="harmony_RNA")

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5, repr.plot.res = 200)
after_umap_s <- DimPlot(integrated_noDBL, reduction = 'harmony_RNA', group.by = 'Sample', pt.size = 1.5, raster = TRUE, cols = my_c)

after_umap_s

ggsave(file="Harmony_RNA_noDBL_sample_UMAP.svg", plot=after_umap_s, width=8, height=4)


options(repr.plot.width = 6, repr.plot.height = 5, repr.plot.res = 200)
after_umap_a <- DimPlot(integrated_noDBL, reduction = 'harmony_RNA', group.by = 'Age', pt.size = 1.5, raster = TRUE, cols = my_c[2:3])

after_umap_a

ggsave(file="Harmony_RNA_noDBL_Age_UMAP.svg", plot=after_umap_a, width=6, height=4)

options(repr.plot.width = 6, repr.plot.height = 5, repr.plot.res = 200)
after_umap_b <- DimPlot(integrated_noDBL, reduction = 'harmony_RNA', group.by = 'Batch', pt.size = 1.5, raster = TRUE, cols = my_c[2:3])

after_umap_b

ggsave(file="Harmony_RNA_noDBL_Batch_UMAP.svg", plot=after_umap_b, width=6, height=4)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated_noDBL, group.by="orig.ident", split.by="orig.ident", reduction="harmony_RNA", shuffle=TRUE, cols=my_c, ncol=3, pt.size=1.5, label=FALSE)

dpl

ggsave(file="Harmony_each_sample_noDBL_UMAP.svg", plot=dpl, width=24, height=12)

### Clustering

In [ ]:
integrated_noDBL

In [ ]:
DefaultAssay(integrated_noDBL) <- 'RNA'

In [ ]:


integrated_noDBL <- integrated_noDBL %>% FindNeighbors(reduction = 'harmonyRNA', dims = 1:13) %>%
                                FindClusters(resolution = c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.1, 1.2, 1.3,
                                                           1.4, 1.5, 1.6))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 300)
grid <- plot_grid(DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.1', cols = my_c, raster = TRUE), 
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.2', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.3', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.4', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.5', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.6', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.7', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.8', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.0.9', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.1', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.1.2', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.1.3', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.1.4', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.1.5', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_RNA", pt.size=1.5, group.by = 'RNA_snn_res.1.6', cols = my_c, raster = TRUE))
grid

ggsave(file="Epithelial_integrated_noDBL_clusters_all_noDBL.svg", plot=grid, width=15, height=30)

In [ ]:
integrated_noDBL <- SetIdent(integrated_noDBL, value = 'RNA_snn_res.0.9')

In [ ]:
integrated_noDBL_predicted.markers <- FindAllMarkers(integrated_noDBL, test.use="wilcox", only.pos=TRUE, max.cells.per.ident=15)

In [ ]:
integrated_noDBL_predicted.markers %>% group_by(cluster) %>%
                                slice_head(n = 10) %>%
                                ungroup() -> top10_predicted

In [ ]:
top10_predicted <- top10_predicted[!duplicated(top10_predicted$gene), ]

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4, repr.plot.res = 300)
dtp <- DotPlot(integrated_noDBL, features = top10_predicted$gene, col.min = 0, col.max = 4, dot.scale = 4, scale=FALSE) + theme(axis.text.x=element_text(size=7, angle = 90),axis.text.y=element_text(size=7), 
                                                                                                 legend.position = 'bottom', legend.direction = 'horizontal', 
                                                                                                  legend.text = element_text(size=7)) + xlab('') + ylab('') & scale_color_distiller(palette = "YlGnBu", direction = 1)

dtp

# ggsave(file="Epithelial_integrated_noDBL_dotplot_predictedid.svg", plot=dtp, width=10, height=6)

In [ ]:
saveRDS(integrated_noDBL, 'Epithelial_integrated_noDBL_multiome_peaks_clusters.rds')

## ATAC

In [ ]:
integrated_noDBL <- readRDS('Epithelial_integrated_noDBL_multiome_peaks_clusters.rds')

In [ ]:
DefaultAssay(integrated_noDBL) <- "peaks"

In [ ]:
integrated_noDBL <- FindTopFeatures(integrated_noDBL, assay = "peaks", min.cutoff = 10)

In [ ]:
integrated_noDBL <- RunTFIDF(integrated_noDBL, assay="peaks")

In [ ]:
integrated_noDBL <- RunSVD(integrated_noDBL, assay="peaks")

In [ ]:
options(repr.plot.width=6, repr.plot.height=4, repr.plot.res = 200)
dpc <- DepthCor(integrated_noDBL, n=50)

dpc

ggsave(file="Merged_depthCorr_noDBL.svg", plot=dpc, width=6, height=4)

In [ ]:
options(repr.plot.width=6, repr.plot.height=4, repr.plot.res = 200)
ebl <- ElbowPlot(integrated_noDBL, reduction = "lsi", n=50)

ebl

ggsave(file="Merged_depthCorr_ebl_noDBL.svg", plot=ebl, width=6, height=4)

In [ ]:
integrated_noDBL <- FindNeighbors(integrated_noDBL, reduction = 'lsi', dims = 2:16, graph.name="snn_peaks")

In [ ]:
integrated_noDBL <- RunUMAP(integrated_noDBL, dims = 2:16, reduction = 'lsi', reduction.name="umap_peaks")

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
dpl <- DimPlot(integrated_noDBL, group.by="Sample", reduction="umap_peaks", pt.size = 1.5, raster = TRUE, cols = my_c, label=FALSE)

dpl 

ggsave(file="Merged_peaks_Sample_UMAP_noDBL.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated_noDBL, group.by="orig.ident", split.by="orig.ident", reduction="umap_peaks", shuffle=TRUE, cols=my_c, ncol=3, pt.size=1.5, raster = TRUE, label=FALSE)

dpl

ggsave(file="Merged_peaks_each_sample_UMAP_noDBL.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=7, repr.plot.height=4)
dpl <- DimPlot(integrated_noDBL, group.by="Batch", reduction="umap_peaks", pt.size = 1.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Merged_peaks_Batch_UMAP_noDBL.svg", plot=dpl, width=7, height=4)

In [ ]:
options(repr.plot.width=7, repr.plot.height=4)
dpl <- DimPlot(integrated_noDBL, group.by="Age", reduction="umap_peaks", pt.size = 1.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Merged_peaks_Age_UMAP_noDBL.svg", plot=dpl, width=7, height=4)

### Integration with Harmony

In [ ]:
library(harmony)

In [ ]:
integrated_noDBL <- RunHarmony(integrated_noDBL, group.by.vars = 'Sample', reduction = 'lsi', assay.use = 'peaks', reduction.save = 'harmonypeaks', project.dim = FALSE)

In [ ]:
# re-compute the UMAP using corrected LSI embeddings
integrated_noDBL <- RunUMAP(integrated_noDBL, dims = 2:16, reduction = 'harmonypeaks', reduction.name = 'harmony_Peaks')

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
dpl <- DimPlot(integrated_noDBL, group.by = 'Sample', pt.size = 1.5, raster=TRUE, cols=my_c, reduction = 'harmony_Peaks')

dpl

ggsave(file="Epithelial_integrated_noDBL_peaks_noDBL_Sample_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated_noDBL, group.by="orig.ident", split.by="orig.ident", reduction="harmony_Peaks", shuffle=TRUE, cols=my_c, ncol=3, pt.size=1.5, raster = TRUE, label=FALSE)

dpl

ggsave(file="Epithelial_integrated_noDBL_peaks_noDBL_each_Sample_UMAP.svg", plot=dpl, width=24, height=12)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)
dpl <- DimPlot(integrated_noDBL, group.by="Batch", reduction="harmony_Peaks", pt.size = 1.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Epithelial_integrated_noDBL_Peaks_noDBL_Batch_UMAP.svg", plot=dpl, width=7, height=5)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)
dpl <- DimPlot(integrated_noDBL, group.by="Age", reduction="harmony_Peaks", pt.size = 1.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Epithelial_integrated_noDBL_Peaks_noDBL_Age_UMAP.svg", plot=dpl, width=7, height=5)

### Clustering

In [ ]:
DefaultAssay(integrated_noDBL) <- 'peaks'

In [ ]:
integrated_noDBL

In [ ]:
integrated_noDBL <- integrated_noDBL %>% FindNeighbors(reduction = 'harmonypeaks', dims = 2:16) %>%
                                FindClusters(resolution = c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 300)
grid <- plot_grid(DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.1', cols = my_c, raster = TRUE), 
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.2', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.3', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.4', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.5', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.6', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.7', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.8', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=1.5, group.by = 'peaks_snn_res.0.9', cols = my_c, raster = TRUE),
                  DimPlot(integrated_noDBL, reduction = "harmony_Peaks", pt.size=1.5, group.by = 'peaks_snn_res.1', cols = my_c, raster = TRUE))
grid

ggsave(file="Epithelial_integrated_noDBL_Peaks_noDBL_clusters_all.svg", plot=grid, width=15, height=30)

In [ ]:
marker_genes <- c('Folr1', 'Clic6', 'Krt18', 'Ttr', 'Dcn', 'Itga8', 'Col3a1', 'Flt1', 'Plvap', 'Pecam1', 'Cd86', 'Tank', 
                  'Cd83', 'Dscam', 'Ptprz1', 'Gria2')

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20, repr.plot.res = 200)
ftp <- FeaturePlot(integrated_noDBL, reduction = 'harmony_Peaks', features = marker_genes, min.cutoff = 0,
                   label.size = 2, ncol = 4, pt.size = 1.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 



In [ ]:
saveRDS(integrated_noDBL, 'Epithelial_integrated_noDBL_multiome_clusters.rds')

# Cell type annotation

In [ ]:
integrated_noDBL <- readRDS('Epithelial_integrated_noDBL_multiome_clusters.rds')

In [ ]:
DefaultAssay(integrated_noDBL) <- 'RNA'

In [ ]:
integrated <- SetIdent(integrated_noDBL, value = 'RNA_snn_res.0.9')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_peaks', 'nFeature_peaks', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

In [ ]:
marker_genes <- c('Folr1', 'Clic6', 'Krt18', 'Dcn', 'Itga8', 'Col3a1', 'Flt1', 'Plvap', 'Pecam1', 'Cd86', 'Tank', 
                  'Cd83', 'Dscam', 'Ptprz1', 'Gria2')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 14, repr.plot.res = 300)
vln <- VlnPlot(object = integrated_noDBL, features = marker_genes, pt.size=0.5, cols = my_c, ncol = 3)

vln

ggsave(file="Epithelial_integrated_noDBL_vln_Cell_type.svg", plot=vln, width=10, height=25)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20, repr.plot.res = 200)
ftp <- FeaturePlot(integrated_noDBL, reduction = 'harmony_RNA', features = marker_genes, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 1.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 



In [ ]:
Epithelial_noDBL_markers = c('Zfp949', 'Gpx3', 'Fam227b', 'Cp',
           'Rian', 'Tenm3', 'Tbc1d1', 'Cidea', 
           'Deup1', 'Cdc20b', 'Ccno', 'Shisa8',
           'Birc5', 'Lockd', 'Mki67', 'Nusap1',
           'Ttr', 'Folr1', 'Clic6', 'Otx2',
           'Pon1', 'Sgcd', 'Gata3', 'Atp10b')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 14, repr.plot.res = 300)
vln <- VlnPlot(object = integrated_noDBL, features = Epithelial_noDBL_markers, pt.size=0.8, cols = my_c, ncol = 4)

vln

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20, repr.plot.res = 200)
ftp <- FeaturePlot(integrated_noDBL, reduction = 'harmony_RNA', features = Epithelial_noDBL_markers, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 1.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 



In [ ]:
options(repr.plot.width = 5, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated_noDBL, group.by = "RNA_snn_res.0.9", pt.size=1.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

ggsave(file="Epithelial_integrated_noDBL_clusters_RNA.svg", plot=dpt, width=7, height=5)

In [ ]:
integrated_noDBL_predicted.markers <- FindAllMarkers(integrated_noDBL, test.use="wilcox", only.pos=TRUE, max.cells.per.ident=15)

In [ ]:
integrated_noDBL_predicted.markers %>% group_by(cluster) %>%
                                slice_head(n = 10) %>%
                                ungroup() -> top10_predicted_noDBL

In [ ]:
top10_predicted_noDBL <- top10_predicted_noDBL[!duplicated(top10_predicted_noDBL$gene), ]

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 4, repr.plot.res = 300)
dtp <- DotPlot(integrated_noDBL, features = top10_predicted$gene, col.min = 0, col.max = 4, dot.scale = 4, scale=FALSE) + theme(axis.text.x=element_text(size=7, angle = 90),axis.text.y=element_text(size=7), 
                                                                                                 legend.position = 'bottom', legend.direction = 'horizontal', 
                                                                                                  legend.text = element_text(size=7)) + xlab('') + ylab('') & scale_color_distiller(palette = "YlGnBu", direction = 1)

dtp

### Clusters

In [ ]:
integrated_noDBL@meta.data$Clusters <- integrated_noDBL@meta.data$RNA_snn_res.0.9

In [ ]:
integrated_noDBL@meta.data$Clusters[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "1")] <- "0" 
integrated_noDBL@meta.data$Clusters[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "2")] <- "0" 
integrated_noDBL@meta.data$Clusters[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "9")] <- "0" 
integrated_noDBL@meta.data$Clusters[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "3")] <- "0" 
integrated_noDBL@meta.data$Clusters[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "6")] <- "0" 
integrated_noDBL@meta.data$Clusters[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "10")] <- "0" 
integrated_noDBL@meta.data$Clusters[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "13")] <- "0" 
integrated_noDBL@meta.data$Clusters[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "7")] <- "0" 
integrated_noDBL@meta.data$Clusters[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "8")] <- "0" 
integrated_noDBL@meta.data$Clusters[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "4")] <- "0" 
integrated_noDBL@meta.data$Clusters[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "5")] <- "0" 
integrated_noDBL@meta.data$Clusters[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "14")] <- "11" 

In [ ]:
integrated_noDBL <- SetIdent(integrated_noDBL, value = 'Clusters')

In [ ]:
integrated_predicted_noDBL.markers <- FindAllMarkers(integrated_noDBL, test.use="wilcox", only.pos=TRUE, max.cells.per.ident=15)

In [ ]:
integrated_predicted_noDBL.markers %>% group_by(cluster) %>%
                                slice_head(n = 10) %>%
                                ungroup() -> top10_predicted_noDBL

In [ ]:
top10_predicted_noDBL <- top10_predicted_noDBL[!duplicated(top10_predicted_noDBL$gene), ]

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4, repr.plot.res = 300)
dtp <- DotPlot(integrated_noDBL, features = top10_predicted_noDBL$gene, col.min = 0, col.max = 4, dot.scale = 4, scale=FALSE) + theme(axis.text.x=element_text(size=7, angle = 90),axis.text.y=element_text(size=7), 
                                                                                                 legend.position = 'bottom', legend.direction = 'horizontal', 
                                                                                                  legend.text = element_text(size=7)) + xlab('') + ylab('') & scale_color_distiller(palette = "YlGnBu", direction = 1)

dtp

# ggsave(file="Epithelial_integrated_dotplot_predictedid.svg", plot=dtp, width=10, height=6)

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated_noDBL, group.by = "Clusters", pt.size=1.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

ggsave(file="Epithelial_integrated_noDBL_clusters_RNA.svg", plot=dpt, width=7, height=5)

In [ ]:
integrated_noDBL$Cell_type <- NULL

## Cell type

In [ ]:
integrated_noDBL@meta.data$RNA_snn_res.0.9 <- as.factor(integrated_noDBL@meta.data$RNA_snn_res.0.9)

In [ ]:
integrated_noDBL@meta.data$Cell_type <- NA

In [ ]:
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "0")] <- "Epithelial cells" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "1")] <- "Epithelial cells" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "2")] <- "Epithelial cells" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "3")] <- "Epithelial cells"
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "4")] <- "Epithelial cells" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "5")] <- "Epithelial cells" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "6")] <- "Epithelial cells" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "7")] <- "Epithelial cells" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "8")] <- "Epithelial cells" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "9")] <- "Epithelial cells" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "10")] <- "Epithelial cells" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "11")] <- "Gata3+" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "12")] <- "Epithelial cells" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "13")] <- "Epithelial cells" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "14")] <- "Gata3+" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "15")] <- "Ciliogenesis" 
integrated_noDBL@meta.data$Cell_type[which(integrated_noDBL@meta.data$RNA_snn_res.0.9 == "16")] <- "Proligerating epithelial cells" 

In [ ]:
integrated_noDBL <- SetIdent(integrated_noDBL, value = 'Cell_type')

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated_noDBL, group.by = "Cell_type", pt.size=1.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

ggsave(file="Epithelial_integrated_noDBL_celltype_RNA.svg", plot=dpt, width=7, height=4)

In [ ]:
integrated_noDBL_predicted.markers <- FindAllMarkers(integrated_noDBL, test.use="wilcox", only.pos=TRUE, max.cells.per.ident=15)

In [ ]:
write.csv(integrated_noDBL_predicted.markers, 'Markers_RNA_Epithelial.csv', row.names = TRUE)

In [ ]:
integrated_noDBL_predicted.markers %>% group_by(cluster) %>%
                                slice_head(n = 10) %>%
                                ungroup() -> top10_predicted_noDBL

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4, repr.plot.res = 300)
dtp <- DotPlot(integrated_noDBL, features = top10_predicted_noDBL$gene, col.min = 0, col.max = 4, dot.scale = 4, scale=FALSE) + theme(axis.text.x=element_text(size=7, angle = 90),axis.text.y=element_text(size=7), 
                                                                                                 legend.position = 'bottom', legend.direction = 'horizontal', 
                                                                                                  legend.text = element_text(size=7)) + xlab('') + ylab('') & scale_color_distiller(palette = "YlGnBu", direction = 1)

dtp

ggsave(file="Epithelial_integrated_noDBL_dotplot_Cell_type.svg", plot=dtp, width=10, height=6)

In [ ]:
Epithelial_noDBL_markers <- c('Ttr', 'Folr1', 'Clic6', 'Otx2', 
                        'Gata3', 'Atp10b', 'Sgcd', 'Maob', 
                        'Deup1', 'Cdc20b', 'Ccno', 'Myb',
                        'Birc5', 'Lockd', 'Mki67', 'Nusap1')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 20, repr.plot.res = 300)
vln <- VlnPlot(object = integrated_noDBL, features = Epithelial_noDBL_markers, pt.size=0, cols = my_c, ncol = 4)

vln

ggsave(file="Epithelial_integrated_noDBL_vln_Cell_type.svg", plot=vln, width=15, height=20)

In [ ]:
integrated_noDBL

In [ ]:
saveRDS(integrated_noDBL, 'Epithelial_integrated_multiome_celltype.rds')

In [ ]:
markers <- c('Ttr', 'Folr1', 'Clic6', 'Otx2', 
                        'Gata3', 'Atp10b', 'Sgcd', 'Maob', 
                        'Deup1', 'Cdc20b', 'Ccno', 'Myb',
                        'Birc5', 'Lockd', 'Mki67', 'Nusap1')

In [ ]:
expression <- as.data.frame(t(integrated_noDBL[["RNA"]]@data[markers,]))


In [ ]:
expression$Cell <- rownames(expression)
expression$Idents <- Idents(integrated_noDBL)

In [ ]:
expression <- reshape2::melt(expression, id.vars = c("Cell","Idents"), measure.vars = markers,
                       variable.name = "Feat", value.name = "Expr")

In [ ]:
expression$color_group <- as.factor((as.numeric(factor(expression$Feat)) - 1) %/% 4)


In [ ]:
options(repr.plot.width = 20, repr.plot.height = 7, repr.plot.res = 100)

# Create a grouping variable for coloring every 4 genes the same
expression$color_group <- as.factor((as.numeric(factor(expression$Feat)) - 1) %/% 4)

c <- ggplot(expression, aes(factor(Feat), Expr, fill = color_group)) +
        geom_violin(scale = "width", adjust = 1, trim = TRUE) +
        scale_y_continuous(expand = c(0, 0), position="right", labels = function(x)
                           c(rep(x = "", times = length(x)-2), x[length(x) - 1], "")) +
        facet_grid(rows = vars(Idents), scales = "free", switch = "y") +
        theme_cowplot(font_size = 12) +
        theme(legend.position = "none", panel.spacing = unit(0, "lines"),
          plot.title = element_text(hjust = 0.5),
              panel.background = element_rect(fill = NA, color = "black"),
              strip.background = element_blank(),
              strip.text = element_text(face = "bold"),
              strip.text.y.left = element_text(angle = 0),
              axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5)) + scale_fill_manual(values = c(my_c[1:3], my_c[5]))

print(c)


ggsave(file="New_Epithelial_noDBL_all_integrated_vln_Cell_type.svg", plot=c, width=20, height=7)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated_noDBL, group.by = "Cell_type", pt.size=1.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

ggsave(file="Epithelial_integrated_noDBL_celltype_RNA.svg", plot=dpt, width=7, height=4)

In [ ]:
DefaultAssay(integrated_noDBL) <- 'peaks'

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated_noDBL, group.by = "Cell_type", pt.size=1.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_Peaks')

dpt

ggsave(file="Epithelial_integrated_noDBL_celltype_peaks.svg", plot=dpt, width=7, height=4)

In [ ]:
saveRDS(integrated_noDBL, 'Epithelial_integrated_noDBL_multiome_celltype.rds')

# Joint vizualization

In [ ]:
integrated_noDBL <- readRDS('Epithelial_integrated_noDBL_multiome_celltype.rds')

In [ ]:
Annotation(integrated_noDBL) <- annotation

In [ ]:
integrated_noDBL@reductions

In [ ]:
integrated_noDBL <- FindMultiModalNeighbors(integrated_noDBL, reduction.list = c('harmonyRNA', 'harmonypeaks'),
                                       dims.list = list(1:13,2:16), verbose=TRUE)

In [ ]:
integrated_noDBL <- RunUMAP(integrated_noDBL, nn.name = "weighted.nn", assay = "RNA", reduction.name = 'umap_multiome',  verbose = TRUE)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated_noDBL, group.by = "Cell_type", pt.size=1.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'umap_multiome')

dpt

ggsave(file="Epithelial_integrated_noDBL_clusters_predictedid_Multiome.svg", plot=dpt, width=7, height=5)

In [ ]:
saveRDS(integrated_noDBL, 'Epithelial_integrated_noDBL_multiome_celltype_joint.rds')

# Marker genes on different umaps assays

In [ ]:
integrated_noDBL <- readRDS('Epithelial_integrated_noDBL_multiome_celltype_joint.rds')

In [ ]:
Epithelial_noDBL_markers <- c('Ttr', 'Folr1', 'Clic6', 'Otx2', 
                        'Gata3', 'Atp10b', 'Sgcd', 'Maob', 
                        'Deup1', 'Cdc20b', 'Ccno', 'Myb',
                        'Birc5', 'Lockd', 'Mki67', 'Nusap1')

## RNA

In [ ]:
DefaultAssay(integrated_noDBL) <- 'RNA'

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Cell_type', cols = my_c, 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 1.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_RNA_umap.svg", plot=dmp, width=8, height=4)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20, repr.plot.res = 200)
ftp <- FeaturePlot(integrated_noDBL, reduction = 'harmony_RNA', features = Epithelial_noDBL_markers, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 1.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


ggsave(file="Harmony_noDBL_geneExpression_RNA_umap.svg", plot=ftp, width=20, height=20)

## Peaks

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Cell_type', cols = my_c, reduction = 'harmony_Peaks', raster = TRUE, 
               pt.size = 1.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_Peaks_umap.svg", plot=dmp, width=8, height=4)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20, repr.plot.res = 200)
ftp <- FeaturePlot(integrated_noDBL, reduction = 'harmony_Peaks', features = Epithelial_noDBL_markers, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 1.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


ggsave(file="Harmony_noDBL_geneExpression_Peaks_umap.svg", plot=ftp, width=20, height=20)

## Joint vizualization

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Cell_type', cols = my_c, reduction = 'umap_multiome', raster = TRUE,
               pt.size = 1.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_Joint_umap.svg", plot=dmp, width=8, height=4)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20, repr.plot.res = 200)
ftp <- FeaturePlot(integrated_noDBL, reduction = 'umap_multiome', features = Epithelial_noDBL_markers, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 1.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


ggsave(file="Harmony_noDBL_geneExpression_Joint_umap.svg", plot=ftp, width=20, height=20)

# Linking peaks to genes

In [ ]:
integrated_noDBL <- readRDS('Epithelial_integrated_noDBL_multiome_celltype_joint.rds')

In [ ]:
library(BSgenome.Mmusculus.UCSC.mm10)

In [ ]:
DefaultAssay(integrated_noDBL) <- 'peaks'

In [ ]:
# first compute the GC content for each peak
integrated_noDBL <- RegionStats(integrated_noDBL, genome = BSgenome.Mmusculus.UCSC.mm10)

In [ ]:
integrated_noDBL

In [ ]:
integrated_noDBL <- LinkPeaks(integrated_noDBL, peak.assay = "peaks", expression.assay = "RNA")

In [ ]:
integrated_noDBL

In [ ]:
saveRDS(integrated_noDBL, 'Epithelial_integrated_noDBL_multiome_celltype_joint_linked.rds')

# Find DA peaks

In [ ]:
integrated_noDBL <- readRDS('Epithelial_integrated_noDBL_multiome_celltype_joint_linked.rds')

## Cell types

In [ ]:
DefaultAssay(integrated_noDBL) <- 'peaks'
Idents(integrated_noDBL) <- "Cell_type"

In [ ]:
da_peaks <- FindAllMarkers(
  object = integrated_noDBL,
  test.use = 'LR',
  min.pct = 0.1
)

In [ ]:
da_peaks <- da_peaks[da_peaks$p_val_adj < 0.05, ]

In [ ]:
write.csv(da_peaks, 'DA_peaks_epithelial.csv')

## Condition

In [ ]:
Idents(integrated_noDBL) <- "Age"

In [ ]:
da_peaks_age <- FindAllMarkers(
  object = integrated_noDBL,
  assay = 'peaks',
  test.use = 'wilcox',
  min.pct = 0.01
)

In [ ]:
dim(da_peaks_age)

In [ ]:
da_peaks_age <- da_peaks_age[da_peaks_age$p_val_adj < 0.05, ]

In [ ]:
write.csv(da_peaks_age, 'DA_peaks_Epithelial_age.csv')

In [ ]:
dim(da_peaks_age)

In [ ]:
p11_peaks <- subset(da_peaks_age, cluster == 'P11')

In [ ]:
dim(p11_peaks)

In [ ]:
p60_peaks <- subset(da_peaks_age, cluster == 'P60')

In [ ]:
dim(p60_peaks)

# Plots for figures

In [ ]:
integrated_noDBL$log10_ngenes <- log10(integrated_noDBL$nFeature_RNA)

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('log10_ngenes'), 
        ncol=4, pt.size=0, col = my_c) + scale_y_continuous(limits = c(0,5)) + scale_fill_manual(values = c(my_c[1:3], my_c[5]))

vln

ggsave(file="noDBL_Epithelial_integrated_all_vln_log10genes_Cell_type.svg", plot=vln, width=14, height=5)


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('pct_mito'), 
        ncol=4, pt.size=0, col = my_c) + scale_fill_manual(values = c(my_c[1:3], my_c[5])) + scale_y_continuous(limits = c(0,20))

vln

ggsave(file="noDBL_Epithelial_integrated_all_vln_pct_mito_Cell_type.svg", plot=vln, width=14, height=5)


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('log10_ngenes'), group.by = 'Sample',
        ncol=4, pt.size=0, col = my_c) + scale_y_continuous(limits = c(0,5))

vln

ggsave(file="noDBL_Epithelial_integrated_all_vln_log10genes_Sample.svg", plot=vln, width=14, height=5)


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('pct_mito'), group.by = 'Sample',
        ncol=4, pt.size=0, col = my_c) + scale_y_continuous(limits = c(0,20))

vln

ggsave(file="noDBL_Epithelial_integrated_all_vln_pct_mito_Sample.svg", plot=vln, width=14, height=5)


In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('nCount_RNA', 'log10_ngenes', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="noDBL_Epithelial_integrated_all_vln_log10genes_pct_mito_all.svg", plot=vln, width=15, height=8)


In [ ]:
DefaultAssay(integrated_noDBL) <- 'RNA'

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('Ttr, eGFP'), group.by = 'batch',
        ncol=4, pt.size=0, col = my_c) 

vln

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('Ttr'), group.by = 'Age',
        ncol=4, pt.size=0, col = my_c[2:3]) 

vln

ggsave(file="noDBL_Epithelial_integrated_all_vln_TTR.svg", plot=vln, width=14, height=5)


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('eGFP'), group.by = 'Age',
        ncol=4, pt.size=1, col = my_c[2:3]) 

vln

ggsave(file="noDBL_Epithelial_integrated_all_vln_eGFP.svg", plot=vln, width=14, height=5)


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 300)

vln <- VlnPlot(integrated_noDBL, features=c('CreERT'), group.by = 'Age',
        ncol=4, pt.size=1, col = my_c[2:3]) 

vln

ggsave(file="noDBL_Epithelial_integrated_all_vln_.svg", plot=vln, width=14, height=5)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE, cols = c(my_c[1:3], my_c[5]), 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 1.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_RNA_umap.svg", plot=dmp, width=8, height=4)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE, cols = c(my_c[1:3], my_c[5]), 
               reduction = 'harmony_peaks', raster = TRUE, pt.size = 1.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_peaks_umap.svg", plot=dmp, width=8, height=4)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE, cols = c(my_c[1:3], my_c[5]), 
               reduction = 'umap_multiome', raster = TRUE, pt.size = 1.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_joint_umap.svg", plot=dmp, width=8, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Age', cols = my_c[2:3], 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 1.5)

dmp

ggsave(file="Harmony_noDBL_Age_RNA_umap.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Age', cols = my_c[2:3], 
               reduction = 'harmony_peaks', raster = TRUE, pt.size = 1.5)

dmp

ggsave(file="Harmony_noDBL_Age_peaks_umap.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Age', cols = my_c[2:3], 
               reduction = 'umap_multiome', raster = TRUE, pt.size = 1.5)

dmp

ggsave(file="Harmony_noDBL_Age_joint_umap.svg", plot=dmp, width=6, height=4)

## Highlight DA peaks

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5, repr.plot.res = 200)

### Cdc20b

In [ ]:
regions <- c('chr13-113034383-113035442')

In [ ]:
ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated_noDBL, region = 'Cdc20b', feature = 'Cdc20b', region.highlight = ranges.show, group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("highlight_noDBL_", 'Cdc20b', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated_noDBL, region = 'Cdc20b', feature = 'Cdc20b', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Cdc20b', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

### Deup1

In [ ]:
regions <- c('chr9-15627536-15628287')

In [ ]:
ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated_noDBL, region = 'Deup1', feature = 'Deup1', region.highlight = ranges.show, group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 2000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("highlight_noDBL_", 'Deup1', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated_noDBL, region = 'Deup1', feature = 'Deup1', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 2000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Deup1', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

### Mki67

In [ ]:
cvp <- CoveragePlot(integrated_noDBL, region = 'Mki67', feature = 'Mki67', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 2000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Mki67', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

### Nusap1

In [ ]:
cvp <- CoveragePlot(integrated_noDBL, region = 'Nusap1', feature = 'Nusap1', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Nusap1', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

### Atp10b

In [ ]:
cvp <- CoveragePlot(integrated_noDBL, region = 'Atp10b', feature = 'Atp10b', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Atp10b', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

### Gata3

In [ ]:
regions <- c('chr2-9878395-9878973', 'chr2-9879176-9879877')

In [ ]:
ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated_noDBL, region = 'Gata3', feature = 'Gata3', region.highlight = ranges.show, group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 10000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("highlight_noDBL_", 'Gata3', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated_noDBL, region = 'Gata3', feature = 'Gata3', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 10000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Gata3', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

## Peaks by Age

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5, repr.plot.res = 200)

### Ifi27

In [ ]:
regions <- c('chr12-103437630-103438002', 'chr12-103433253-103434544', 'chr12-103443434-103443757')

In [ ]:
ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated_noDBL, region = 'Ifi27', feature = 'Ifi27', region.highlight = ranges.show, group.by = 'Age',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream =5000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Age_highlight_noDBL_", 'Ifi27', "_igv_tracks_epi.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated_noDBL, region = 'Ifi27', feature = 'Ifi27', group.by = 'Age',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream =5000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Age_no_highlight_noDBL_", 'Ifi27', "_igv_tracks_epi.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

### Slit3

In [ ]:
regions <- c('chr11-35120588-35121924', 'chr11-35231492-35232745', 'chr11-35223456-35224579',
            'chr11-35338003-35338825', 'chr11-35327794-35328481', 'chr11-35454135-35455353', 
            'chr11-35452525-35452732', 'chr11-35407481-35408220', 'chr11-35558957-35559722', 'chr11-35586889-35587755',
            'chr11-35663472-35664337', 'chr11-35664997-35665253', 'chr11-35648968-35649440')

In [ ]:
ranges.show <- StringToGRanges(regions)

In [ ]:
cvp <- CoveragePlot(integrated_noDBL, region = 'Slit3', feature = 'Slit3', region.highlight = ranges.show, group.by = 'Age',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream =1000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Age_highlight_noDBL_", 'Slit3', "_igv_tracks_epi.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated_noDBL, region = 'Slit3', feature = 'Slit3', group.by = 'Age',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream =1000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Age_no_highlight_noDBL_", 'Slit3', "_igv_tracks_epi.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

# Xist counts to try to separate male and female cells 

In [ ]:
integrated_noDBL <- readRDS('Epithelial_integrated_noDBL_multiome_celltype_joint_linked.rds')

In [ ]:
DefaultAssay(integrated_noDBL) <- 'RNA'

In [ ]:
integrated_noDBL

In [ ]:
xist_counts <- GetAssayData(integrated_noDBL, assay = "RNA", layer = "counts")[ "Xist",]

In [ ]:
integrated_noDBL$'Xist-counts' <- xist_counts

In [ ]:
integrated_noDBL$'Female' <- integrated_noDBL$`Xist-counts` > 0

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Female', cols = my_c[2:3], 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 1.5)

dmp

ggsave(file="Gender_inference_Harmony_noDB_RNA_umap.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Female', cols = my_c[2:3], 
               reduction = 'harmony_peaks', raster = TRUE, pt.size = 1.5)

dmp

ggsave(file="Gender_inference_Harmony_noDBL_peaks_umap.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated_noDBL, label = FALSE,group.by='Female', cols = my_c[2:3], 
               reduction = 'umap_multiome', raster = TRUE, pt.size = 1.5)

dmp

ggsave(file="Gender_inference_Harmony_noDBL_joint_umap.svg", plot=dmp, width=6, height=4)

# Session info

In [ ]:
print(sessioninfo::session_info())